In [17]:
import pandas as pd 
import geopandas as gpd 

import duckdb

duckdb.sql('''
install spatial;
load spatial;''')

duckdb.sql('SET geometry_always_xy = true')

In [2]:
duckdb.sql('''
CREATE OR REPLACE TABLE filosofi AS 
select * 
from read_parquet('https://www.data.gouv.fr/api/1/datasets/r/55432374-a91d-43d0-923d-4514dc3eb951');
''')

duckdb.sql('''
ALTER TABLE filosofi
ALTER COLUMN geometry TYPE GEOMETRY USING ST_GeomFromText(ST_AsText(geometry));           

CREATE INDEX my_idx ON filosofi USING RTREE (geometry);
''')


In [3]:
vars = [
'ind', 'men', 'men_pauv', 'men_1ind', 'men_5ind',
'men_prop', 'men_fmp', 'ind_snv', 'men_surf', 'men_coll', 'men_mais',
'log_av45', 'log_45_70', 'log_70_90', 'log_ap90', 'log_inc', 'log_soc',
'ind_0_3', 'ind_4_5', 'ind_6_10', 'ind_11_17', 'ind_18_24', 'ind_25_39',
'ind_40_54', 'ind_55_64', 'ind_65_79', 'ind_80p', 'ind_inc'
]



In [4]:
hubblo = "st_buffer(ST_Point (3756295, 2889313),1000)"
agg = ",\n".join([f"COALESCE(sum({v}*weight), 0) as {v}" for v in vars])

query = '''     
WITH intersected AS (
    SELECT *, st_area(st_intersection(geometry, {})) / st_area(geometry) as weight
    from filosofi
    where ST_Intersects(geometry, {})
)
SELECT
    'hubblo' AS unit, {}
FROM intersected; '''.format(hubblo, hubblo, agg)
duckdb.sql(query)

┌─────────┬───────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬───────────────────┬───────────────────┐
│  unit   │        ind        │        men        │     men_pauv      │     men_1ind      │      men_5ind      │     men_prop      │      men_fmp      │      ind_snv       │      men_surf      │     men_coll      │      men_mais      │      log_av45      │     log_45_70     │     log_70_90     │      log_ap90      │      log_inc      │      log_soc      │      ind_0_3       │      ind_4_5      │      ind_6_10      

In [5]:
commune = gpd.read_file('commune_francemetro_2025.gpkg')[['code', 'libelle', 'geometry']].to_crs(3035).to_parquet('commune_francemetro_2025.geoparquet')

In [6]:
duckdb.sql('''
CREATE OR REPLACE TABLE commune_bas AS 
select * 
from read_parquet('commune_francemetro_2025.geoparquet');
''')

duckdb.sql('''

ALTER TABLE commune_bas
ALTER COLUMN geometry TYPE GEOMETRY USING ST_GeomFromText(ST_AsText(geometry));  
           
CREATE INDEX spatial_index ON commune_bas USING RTREE (geometry);
''')

In [9]:
agg = ",\n".join([f"sum({v}*weight) as {v}" for v in vars])

duckdb.sql(f'''
COPY (
select * 
from commune_bas as c, 
(select code, {agg} from ( 
        SELECT  *, st_area(st_intersection(c.geometry, f.geometry)) / st_area(f.geometry) as weight FROM commune_bas as c, filosofi as f         where st_intersects(c.geometry, f.geometry)               )  group by code) as cc
        where c.code == cc.code   )
TO 'commune_2025_carreaux_2021.parquet'
(FORMAT 'parquet');
''')

In [7]:
duckdb.sql('''
CREATE OR REPLACE TABLE commune AS 
select * 
from read_parquet('commune_2025_carreaux_2021.parquet');
''')

duckdb.sql('''

ALTER TABLE commune
ALTER COLUMN geometry TYPE GEOMETRY USING ST_GeomFromText(ST_AsText(geometry));  
           
CREATE INDEX spatial_inde ON commune USING RTREE (geometry);
''')

In [8]:
agg_commune = ",\n".join([f"{v}" for v in vars])
#print(agg_commune)
hubblo = "st_buffer(ST_Point (3756295, 2889313),1000)"
duckdb.sql(f'''
select concat(code, '-', libelle)as unit , {agg_commune}  
from commune 
where   ST_Intersects(geometry, ST_centroid({hubblo}))                    ;
''')


┌─────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬──────────────────┬───────────────────┬───────────────────┬───────────────────┬──────────────────┬──────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│    unit     │        ind         │        men         │      men_pauv      │      men_1ind      │     men_5ind      │     men_prop     │      men_fmp      │      ind_snv      │     men_surf      │     men_coll     │     men_mais     │     log_av45      │     log_45_70      │     log_70_90      │      log_ap90      │      log_inc       │      log_soc       │      ind_0_3      │      ind_4_5      │     ind_6_

In [21]:
vars = ['ind', 'men', 'men_pauv', 'men_1ind', 'men_5ind', 'men_prop', 'men_fmp', 'ind_snv', 'men_surf', 'men_coll', 'men_mais', 'log_av45', 'log_45_70', 'log_70_90', 'log_ap90', 'log_inc', 'log_soc', 'ind_0_3', 'ind_4_5', 'ind_6_10', 'ind_11_17', 'ind_18_24', 'ind_25_39', 'ind_40_54', 'ind_55_64', 'ind_65_79', 'ind_80p', 'ind_inc']

agg = ",\n".join([f"COALESCE(sum({v}*weight), 0) as {v}" for v in vars])
agg_commune = ",\n".join([f"COALESCE({v},0) as {v}" for v in vars])
query = '''     
WITH intersected AS (
    SELECT *, st_area(st_intersection(geometry, {})) / st_area(geometry) as weight
    from filosofi
    where ST_Intersects(geometry, {})
)
SELECT
    'hubblo' AS unit, {}, ST_AsText(ST_Transform({}, 'EPSG:3035', 'EPSG:{}')) as geometry
FROM intersected

UNION 

select concat(code, '-', libelle)as unit , {}  , ST_AsText(ST_Transform(geometry, 'EPSG:3035', 'EPSG:{}'))  as geometry
from commune 
where   ST_Intersects(geometry, ST_centroid({}))    
'''

x = 1.44193756295;
y = 43.596;
radius = 200;
epsg = 4326
hubblo = f"(st_buffer(ST_Transform(ST_Point({x}, {y}),'EPSG:4326', 'EPSG:3035'), {radius}))"
duckdb.sql(query.format(hubblo, hubblo, agg, hubblo, epsg, agg_commune, epsg, hubblo))


┌────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬───────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┬────────────────────┬─────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────